# V24 score-first tracking: Kaggle evaluation

Frozen four-arm evaluation of V19, E016 detections with Atabey relinking, the E016 native graph, and the bounded V24.2 interior-orphan shadow. This notebook performs inference and official evaluation only. It does not train, tune thresholds, construct hybrids, create submissions, or mutate production graphs.

Attach the Biohub competition data, `pilkwang/biohub-tracking-support-pack-50ep-v1`, and a private dataset containing the frozen E016 checkpoint plus its `config.json`. Enable a Kaggle GPU and Internet before running.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

BRANCH = "v24-score-first-tracking"
EXPECTED_COMMIT = "d308d7bce5cfe8f9ad3777eae3e3029855ec14ac"
ROOT = Path("/tmp/Atabey")

if not ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "https://github.com/drosadocastro-bit/Atabey.git",
            str(ROOT),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)

subprocess.run(
    ["git", "-C", str(ROOT), "checkout", "--detach", EXPECTED_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = f"{ROOT / 'src'}:{ROOT / 'scripts'}"
sys.path.insert(0, str(ROOT / "src"))
print("Atabey commit verified:", actual_commit)

In [ ]:
pinned_official_packages = [
    "git+https://github.com/royerlab/tracksdata.git@39dccf3a243e44274759468cb31b2ad9e7fc1d09",
    "git+https://github.com/royerlab/kaggle-cell-tracking-competition.git@075fc5f5a52d11077f9dc2b074644618f26939e2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pinned_official_packages],
    check=True,
 )

official_runtime_packages = [
    "bidict>=0.23.1",
    "blosc2",
    "dask",
    "geff>=1.1.3.1.1",
    "ilpy>=0.5.1",
    "imagecodecs",
    "numba",
    "numcodecs>=0.13",
    "numpy>2",
    "polars>=1.36.0",
    "psygnal>=0.14.0",
    "pyarrow",
    "rich",
    "rustworkx>=0.17.1",
    "scikit-image>=0.24.0",
    "sqlalchemy>=2",
    "tqdm",
    "typing-extensions",
    "zarr>=3.0.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *official_runtime_packages],
    check=True,
 )

import numpy as np
import scipy
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before V24 evaluation"

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
EXPECTED_CHECKPOINT_SHA256 = (
    "02e1d65756c3dc5928f68a66a8b0ef99be2a6905fa7bc017aa1d87dbe632fd03"
 )

train_candidates = []
for pattern in ("*/train", "*/*/train", "*/*/*/train"):
    for candidate in INPUT_ROOT.glob(pattern):
        if (
            candidate.is_dir()
            and len(list(candidate.glob("*.zarr"))) == 199
            and len(list(candidate.glob("*.geff"))) == 199
        ):
            train_candidates.append(candidate)
train_candidates = sorted(set(train_candidates))
assert len(train_candidates) == 1, f"Expected one 199-sample train directory: {train_candidates}"
TRAIN_DIR = train_candidates[0]

# Search auxiliary datasets recursively, but never traverse competition volumes.
auxiliary_roots = sorted(
    root
    for root in INPUT_ROOT.iterdir()
    if root.is_dir() and not TRAIN_DIR.is_relative_to(root)
 )
assert auxiliary_roots, "No auxiliary Kaggle input roots found"

predictor_candidates = []
for root in auxiliary_roots:
    predictor_candidates.extend(root.rglob("predict_unet_transformer.py"))
predictor_candidates = sorted(
    path for path in set(predictor_candidates) if path.parent.name == "scripts"
 )
support_candidates = sorted({path.parents[1] for path in predictor_candidates})
assert len(support_candidates) == 1, (
    f"Expected one E016 support repository under {auxiliary_roots}: "
    f"{support_candidates}"
 )
SUPPORT_REPO = support_candidates[0]

weight_candidates = []
for root in auxiliary_roots:
    weight_candidates.extend(root.rglob("edge_predictor_best.pth"))
matching_weights = []
for candidate in sorted(set(weight_candidates)):
    if not (candidate.parent / "config.json").exists():
        continue
    if hashlib.sha256(candidate.read_bytes()).hexdigest() == EXPECTED_CHECKPOINT_SHA256:
        matching_weights.append(candidate)
if len(matching_weights) != 1:
    discovered = [(str(path), hashlib.sha256(path.read_bytes()).hexdigest()) for path in sorted(set(weight_candidates))]
    raise AssertionError(f"Expected one exact frozen checkpoint with SHA-256 {EXPECTED_CHECKPOINT_SHA256}; discovered: {discovered}")
WEIGHTS = matching_weights[0]

checkpoint_config = json.loads((WEIGHTS.parent / "config.json").read_text(encoding="utf-8"))
assert checkpoint_config["window_size"] == 2
assert checkpoint_config["downsample"] == [1, 4, 4]
assert checkpoint_config["unet_out_channels"] == 32
assert checkpoint_config["pool_kernel_um"] == 5.0

print("Train:", TRAIN_DIR)
print("Auxiliary roots:", auxiliary_roots)
print("Support:", SUPPORT_REPO)
print("Checkpoint:", WEIGHTS)
print("Checkpoint SHA-256:", EXPECTED_CHECKPOINT_SHA256)

In [ ]:
focused_tests = [
    ROOT / "tests/test_v24_score_first_tracking_preregistration.py",
    ROOT / "tests/test_unet_graph.py",
    ROOT / "tests/test_official_tracking_metric.py",
]
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", *map(str, focused_tests)],
    check=True,
    env=RUN_ENV,
 )
print("V24 contract, graph conversion, and official metric tests passed")

## Execution gate

The notebook defaults to the three-sample smoke. The frozen 27-sample run may be enabled only after the smoke completes with deterministic replay and its outputs are reviewed. Smoke outcomes cannot change the checkpoint, thresholds, arms, or cohort contract.

In [ ]:
RUN_MODE = "full_27"
AUTHORIZE_FULL_27 = True

assert RUN_MODE in {"smoke", "full_27"}
if RUN_MODE == "full_27":
    assert AUTHORIZE_FULL_27, "Review a passing smoke before authorizing the frozen full-27 run"

SAMPLE_SELECTOR = "smoke" if RUN_MODE == "smoke" else "all"
OUTPUT_DIR = Path(f"/kaggle/working/v24_score_first_tracking_v24_2_{RUN_MODE}")
CONTRACT_PATH = ROOT / "tests/fixtures/v24_score_first_tracking.json"
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
boundaries = contract["boundaries"]
assert boundaries["runner_implemented"] is True
assert boundaries["hybrid_enabled"] is False
assert boundaries["threshold_tuning"] is False
assert boundaries["model_retraining"] is False
assert boundaries["full_199_authorized"] is False
assert boundaries["submission_authorized"] is False
assert boundaries["production_graph_mutation"] is False

print("Run mode:", RUN_MODE)
print("Sample selector:", SAMPLE_SELECTOR)
print("Output:", OUTPUT_DIR)

In [ ]:
command = [
    sys.executable,
    "-u",
    str(ROOT / "scripts/run_v24_score_first_tracking.py"),
    "--train-dir",
    str(TRAIN_DIR),
    "--support-repo",
    str(SUPPORT_REPO),
    "--weights",
    str(WEIGHTS),
    "--contract",
    str(CONTRACT_PATH),
    "--output-dir",
    str(OUTPUT_DIR),
    "--sample-ids",
    SAMPLE_SELECTOR,
    "--unet-batch-size",
    "4",
    "--resume",
    "--verify-determinism",
]

started = time.time()
subprocess.run(command, check=True, env=RUN_ENV)
elapsed_seconds = time.time() - started
print("V24 run elapsed hours:", elapsed_seconds / 3600.0)

In [ ]:
summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
assert summary["determinism_verified"] is True
assert summary["assignment_enabled"] is False
assert summary["hybrid_enabled"] is False
assert summary["production_graph_mutation"] is False

if RUN_MODE == "smoke":
    assert summary["decision"] == "SMOKE_OR_PARTIAL_COMPLETE"
    assert summary["sample_count"] == 3
    assert summary["complete_cohort"] is False
    assert summary["full_199_authorized"] is False
    expected_samples = {
        "44b6_5f15d135",
        "44b6_74d0c52e",
        "6bba_3c5691b6",
    }
    actual_samples = {path.stem for path in (OUTPUT_DIR / "samples").glob("*.json")}
    assert actual_samples == expected_samples, actual_samples
else:
    contracted_outcomes = set(contract["outcomes"])
    assert summary["sample_count"] == 27
    assert summary["complete_cohort"] is True
    assert summary["decision"] in contracted_outcomes
    assert summary["full_199_authorized"] is (
        summary["decision"] == "GO_TO_FULL_199_SCORE_VALIDATION"
    )

print("Decision:", summary["decision"])
for arm, arm_summary in summary["summaries"].items():
    print(arm, arm_summary["overall"])

In [ ]:
BUNDLE = Path("/kaggle/working/v24_score_first_tracking_outputs")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR, BUNDLE / "run")
shutil.copy2(CONTRACT_PATH, BUNDLE / CONTRACT_PATH.name)

run_record = {
    "mode": RUN_MODE,
    "atabey_commit": EXPECTED_COMMIT,
    "checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "support_predictor_sha256": summary["provenance"]["predictor_sha256"],
    "elapsed_seconds": elapsed_seconds,
    "decision": summary["decision"],
    "no_training": True,
}
(BUNDLE / "notebook_run_record.json").write_text(
    json.dumps(run_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
 )
archive = shutil.make_archive(str(BUNDLE), "zip", BUNDLE)
print("Download:", archive)

## Interpretation boundary

A passing smoke authorizes only the frozen 27-sample run after human review. The complete 27-sample decision contract controls whether full-199 score validation is authorized. Neither stage authorizes training, threshold tuning, hybridization, submission, or production graph mutation.